<a href="https://colab.research.google.com/github/Littlederik/pi/blob/main/torch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from torch import nn
import matplotlib.pyplot as plt

In [ ]:
weight=0.7
bias=0.2

start=0
end=1
step=0.02
x=torch.arange(start,end,step).unsqueeze(dim=1)
print(x.shape)
y=weight*x +bias
print(x)

In [ ]:
train_split=int(0.8*len(x))
x_train,y_train=x[:train_split],y[:train_split]
x_test,y_test=x[train_split:],y[train_split:]
print(len(x_train),len(x_test),len(y_train),len(y_test))

In [ ]:
def plot_prediction(train_data=x_train,train_label=y_train,test_data=x_test,test_label=y_test,predictions=None):
  plt.figure(figsize=(10,7))
  plt.scatter(train_data,train_label,c="b",s=4,label="Training data")
  plt.scatter(test_data,test_label,c="g",s=4,label="testing data")

  if predictions is not None:
    plt.scatter(test_data,predictions,c="r",s=4,label="predictions")
  plt.legend(prop={"size":14})

In [ ]:
plot_prediction()

In [ ]:
class linearregressionmodel(nn.Module):
  def __init__(self):
    super().__init__()
    self.weights=nn.Parameter(torch.randn(1,dtype=torch.float),requires_grad=True)
    self.bias=nn.Parameter(torch.randn(1,dtype=torch.float),requires_grad=True)

  def forward(self,x:torch.Tensor) -> torch.Tensor:
    return self.weights * x +self.bias

In [ ]:
torch.manual_seed(42)
model_0=linearregressionmodel()
list(model_0.parameters())

In [ ]:
# List named parameters
model_0.state_dict()

In [ ]:
loss_fn=nn.L1Loss()

optimizer=torch.optim.SGD(params=model_0.parameters(),lr=0.01)

In [ ]:
epochs=200
train_loss_values=[]
test_loss_values=[]
epoch_count=[]

for epoch in range(epochs):
  model_0.train
  y_pred=model_0(x_train)
  loss=loss_fn(y_pred,y_train)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  model_0.eval()

  with torch.inference_mode():
    test_pred=model_0(x_test)
    test_loss=loss_fn(test_pred,y_test.type(torch.float))

    if epoch %10==0:
      epoch_count.append(epoch)
      train_loss_values.append(loss.detach().numpy())
      test_loss_values.append(test_loss.detach().numpy())

      print(f"Epoch: {epoch} | MAE train loss: {loss} | MAE test loss: {test_loss}")


In [ ]:
plt.plot(epoch_count,train_loss_values,label="train_loss")
plt.plot(epoch_count,test_loss_values,label="test_loss")
plt.title("training and test loss curves")
plt.xlabel("epochs")
plt.ylabel("loss")
plt.legend();

In [ ]:
# Find our model's learned parameters
print("The model learned the following values for weights and bias:")
print(model_0.state_dict())
print("\nAnd the original values for weights and bias are:")
print(f"weights: {weight}, bias: {bias}")

In [ ]:
# 1. Set the model in evaluation mode
model_0.eval()

# 2. Setup the inference mode context manager
with torch.inference_mode():
  # 3. Make sure the calculations are done with the model and data on the same device
  # in our case, we haven't setup device-agnostic code yet so our data and model are
  # on the CPU by default.
  # model_0.to(device)
  # X_test = X_test.to(device)
  y_preds = model_0(x_test)
y_preds

In [ ]:
plot_prediction(predictions=y_preds)

In [ ]:
from pathlib import Path

# 1. Create models directory
MODEL_PATH = Path("models")
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# 2. Create model save path
MODEL_NAME = "01_pytorch_workflow_model_0.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME

# 3. Save the model state dict
print(f"Saving model to: {MODEL_SAVE_PATH}")
torch.save(obj=model_0.state_dict(), # only saving the state_dict() only saves the models learned parameters
           f=MODEL_SAVE_PATH)

In [ ]:
# Check the saved file path
!ls -l models/01_pytorch_workflow_model_0.pth

In [ ]:
# Instantiate a new instance of our model (this will be instantiated with random weights)
loaded_model_0 = linearregressionmodel()

# Load the state_dict of our saved model (this will update the new instance of our model with trained weights)
loaded_model_0.load_state_dict(torch.load(f=MODEL_SAVE_PATH))

In [ ]:
# 1. Put the loaded model into evaluation mode
loaded_model_0.eval()

# 2. Use the inference mode context manager to make predictions
with torch.inference_mode():
    loaded_model_preds = loaded_model_0(x_test) # perform a forward pass on the test data with the loaded model

In [ ]:
# Compare previous model predictions with loaded model predictions (these should be the same)
y_preds == loaded_model_preds